In [2]:
import pandas as pd
rents = pd.read_csv(
    "rtb_rents.csv",
    low_memory=False,         # Processes entire file at once (slightly more memory, fewer warnings)         # Resolve mixed-type warning
)

In [12]:
COUNTY_MAP = {
    "Carlow County Council": "Carlow",
    "Cavan County Council *": "Cavan",
    "Clare County Council": "Clare",
    "Cork City Council": "Cork",
    "Cork County Council": "Cork",
    "Donegal County Council": "Donegal",
    "Dublin City Council": "Dublin",
    "DLR County Council": "Dublin",
    "Fingal County Council 1": "Dublin",
    "Galway City Council": "Galway",
    "Galway County Council": "Galway",
    "Kerry County Council": "Kerry",
    "Kildare County Council": "Kildare",
    "Kilkenny County Council": "Kilkenny",
    "Laois County Council": "Laois",
    "Leitrim County Council": "Leitrim",
    "Limerick City and County Council": "Limerick",
    "Longford County Council": "Longford",
    "Louth County Council": "Louth",
    "Mayo County Council": "Mayo",
    "Meath County Council": "Meath",
    "Monaghan County Council": "Monaghan",
    "Offaly County Council": "Offaly",
    "Roscommon County Council": "Roscommon",
    "Sligo County Council": "Sligo",
    "Tipperary County Council": "Tipperary",
    "Waterford City and County Council": "Waterford",
    "Westmeath County Council": "Westmeath",
    "Wexford County Council": "Wexford",
    "Wicklow County Council": "Wicklow",
}

COUNTIES = {
    "Carlow","Cavan","Clare","Cork","Donegal","Dublin","Galway","Kerry","Kildare",
    "Kilkenny","Laois","Leitrim","Limerick","Longford","Louth","Mayo","Meath",
    "Monaghan","Offaly","Roscommon","Sligo","Tipperary","Waterford","Westmeath",
    "Wexford","Wicklow"
}

def standardize_county(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s in COUNTY_MAP:
        return COUNTY_MAP[s]
    s = s.replace(" County Council", "").replace(" City Council", "").strip()
    return s

def parse_time_period(tp):
    m = re.match(r"(\d{4})Q([1-4])", str(tp).strip())
    if not m:
        return np.nan, np.nan
    return int(m.group(1)), int(m.group(2))

def month_to_quarter(x):
    s = str(x).strip()
    m = re.match(r"(\d{4})(\d{2})", s)
    if not m:
        return np.nan, np.nan
    year = int(m.group(1))
    month = int(m.group(2))
    quarter = (month - 1) // 3 + 1
    return year, quarter

def minmax_score(s):
    s = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() == 0:
        return pd.Series(np.nan, index=s.index)
    mn, mx = s.min(), s.max()
    if pd.isna(mn) or pd.isna(mx) or mn == mx:
        return pd.Series(50.0, index=s.index)
    return 100 * (s - mn) / (mx - mn)

In [7]:
rents = pd.read_csv(
    "rtb_rents.csv",
    low_memory=False,         # Processes entire file at once (slightly more memory, fewer warnings)         # Resolve mixed-type warning
)
rents.columns = [str(c).strip() for c in rents.columns]

rents["area_name"] = rents["Location"].map(standardize_county)
rents = rents[rents["area_name"].isin(COUNTIES)].copy()

# keep broad county rows only
if "Number of Bedrooms" in rents.columns:
    rents = rents[rents["Number of Bedrooms"].astype(str).str.contains("All bedrooms", na=False)].copy()

if "Property Type" in rents.columns:
    rents = rents[rents["Property Type"].astype(str).str.contains("All property types", na=False)].copy()

rents["year"] = pd.to_numeric(rents["Year"], errors="coerce")
rents["rent_level"] = pd.to_numeric(rents["VALUE"], errors="coerce")

rents = rents.dropna(subset=["area_name", "year", "rent_level"])
rents = rents.groupby(["area_name", "year"], as_index=False)["rent_level"].mean()

rents_q = []
for _, row in rents.iterrows():
    for q in [1, 2, 3, 4]:
        rents_q.append({
            "area_name": row["area_name"],
            "year": int(row["year"]),
            "quarter": q,
            "time_period": f"{int(row['year'])}Q{q}",
            "rent_level": row["rent_level"]
        })

rents_q = pd.DataFrame(rents_q)
rents_q.head()

,area_name,year,quarter,time_period,rent_level
0,Carlow,2008,1,2008Q1,748.48
1,Carlow,2008,2,2008Q2,748.48
2,Carlow,2008,3,2008Q3,748.48
3,Carlow,2008,4,2008Q4,748.48
4,Carlow,2009,1,2009Q1,689.34


In [8]:
rents_q.shape

(1768, 5)

In [16]:
# make sure this is at the TOP of your notebook (once only)
import pandas as pd
import numpy as np
import re

# ---- load signals ----
signals = pd.read_csv(
    "signals_2.csv",
    skiprows=1,             # skip the extra "signals" row
    encoding="utf-8-sig",   # remove BOM issue
    low_memory=False
)

# clean column names
signals.columns = [str(c).strip().replace("\ufeff", "") for c in signals.columns]

print("Columns:", signals.columns.tolist())
print(signals.head())

# ---- filter to counties ----
signals = signals[signals["area_level"].astype(str).str.lower() == "county"].copy()

# standardize county names
signals["area_name"] = signals["area_name"].map(standardize_county)

# ---- parse time ----
def parse_time_period(tp):
    m = re.match(r"(\d{4})Q([1-4])", str(tp).strip())
    if not m:
        return np.nan, np.nan
    return int(m.group(1)), int(m.group(2))

parsed = signals["time_period"].map(parse_time_period)
signals["year"] = [x[0] for x in parsed]
signals["quarter"] = [x[1] for x in parsed]

# ---- numeric conversion ----
for c in ["rent_growth", "population_growth", "housing_completions"]:
    signals[c] = pd.to_numeric(signals[c], errors="coerce")

# ---- final shape ----
signals = signals[[
    "area_name", "year", "quarter", "time_period",
    "rent_growth", "population_growth", "housing_completions"
]].copy()

# ---- check ----
print(signals.head())
print(signals.shape)

Columns: ['area_name', 'area_level', 'rent_growth', 'population_growth', 'housing_completions', 'time_period']
   area_name area_level  rent_growth  population_growth  housing_completions  \
0       Mayo     county    12.538864           2.016129           555.934208   
1  Tipperary     county    11.971680           1.554692           369.658378   
2   Limerick     county    13.607647           1.381852           560.241862   
3     Carlow     county    12.420807           2.180685           717.817791   
4      Sligo     county    12.957081           0.413793           431.541413   

  time_period  
0      2025Q3  
1      2025Q3  
2      2025Q3  
3      2025Q3  
4      2025Q3  
   area_name  year  quarter time_period  rent_growth  population_growth  \
0       Mayo  2025        3      2025Q3    12.538864           2.016129   
1  Tipperary  2025        3      2025Q3    11.971680           1.554692   
2   Limerick  2025        3      2025Q3    13.607647           1.381852   
3     Carlow

In [17]:
ppi = pd.read_csv(
    "Property_price_index.csv",
    low_memory=False,         # Processes entire file at once (slightly more memory, fewer warnings)         # Resolve mixed-type warning
)

ppi.columns = [str(c).strip() for c in ppi.columns]

print(ppi.columns.tolist())
print(ppi.head())

month_col = "TLIST(M1)"
location_col = "Type of Residential Property"

keep_locations = {
    "National - all residential properties": "ppi_national_all",
    "Dublin - all residential properties": "ppi_dublin_all",
    "National excluding Dublin - all residential properties": "ppi_nat_ex_dublin_all",
}

ppi = ppi[ppi[location_col].isin(keep_locations.keys())].copy()
ppi["feature_name"] = ppi[location_col].map(keep_locations)

parsed = ppi[month_col].map(month_to_quarter)
ppi["year"] = [x[0] for x in parsed]
ppi["quarter"] = [x[1] for x in parsed]

ppi["VALUE"] = pd.to_numeric(ppi["VALUE"], errors="coerce")
ppi = ppi.dropna(subset=["year", "quarter", "VALUE"])

ppi["year"] = ppi["year"].astype(int)
ppi["quarter"] = ppi["quarter"].astype(int)
ppi["time_period"] = ppi["year"].astype(str) + "Q" + ppi["quarter"].astype(str)

ppi_q = (
    ppi.groupby(["year", "quarter", "time_period", "feature_name"], as_index=False)["VALUE"]
    .mean()
    .pivot_table(
        index=["year", "quarter", "time_period"],
        columns="feature_name",
        values="VALUE",
        aggfunc="mean"
    )
    .reset_index()
)

ppi_q.columns.name = None
ppi_q.head()

['STATISTIC', 'Statistic Label', 'TLIST(M1)', 'Month', 'C02803V03373', 'Type of Residential Property', 'UNIT', 'VALUE']
  STATISTIC                   Statistic Label  TLIST(M1)         Month  \
0  HPM06C01  Residential Property Price Index     200501  2005 January   
1  HPM06C01  Residential Property Price Index     200501  2005 January   
2  HPM06C01  Residential Property Price Index     200501  2005 January   
3  HPM06C01  Residential Property Price Index     200501  2005 January   
4  HPM06C01  Residential Property Price Index     200501  2005 January   

  C02803V03373           Type of Residential Property                 UNIT  \
0            -  National - all residential properties  Base Jan 2005 = 100   
1           01                      National - houses  Base Jan 2005 = 100   
2           02                  National - apartments  Base Jan 2005 = 100   
3           05    Dublin - all residential properties  Base Jan 2005 = 100   
4           06                        Dublin 

,year,quarter,time_period,ppi_dublin_all,ppi_nat_ex_dublin_all,ppi_national_all
0,2005,1,2005Q1,61.580000,60.220000,60.720000
1,2005,2,2005Q2,35.733333,35.000000,35.255556
2,2005,3,2005Q3,37.600000,37.300000,37.377778
3,2005,4,2005Q4,40.444444,38.355556,39.100000
4,2006,1,2006Q1,33.333333,31.433333,32.158333


In [25]:
import pandas as pd
import numpy as np
import re

arrears_raw = pd.read_csv("Loan_arrears.csv", header=0, low_memory=False)
arrears_raw.columns = [str(c).strip().replace("\ufeff", "") for c in arrears_raw.columns]

print("Shape:", arrears_raw.shape)
print("Columns:", arrears_raw.columns.tolist()[:20])
arrears_raw.head(5)

Shape: (43, 199)
Columns: ['Local Authority', 'Q4 2015', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Q1 2016', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Q2 2016', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Q3 2016']


,Local Authority,Q4 2015,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Q1 2016,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Q2 2016,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Q3 2016,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Q4 2016,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Q1 2017,Unnamed: 32,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36,Q2 2017,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42,Q3 2017,Unnamed: 44,Unnamed: 45,Unnamed: 46,Unnamed: 47,Unnamed: 48,Q4 2017,Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53,Unnamed: 54,Q1 2018,Unnamed: 56,Unnamed: 57,Unnamed: 58,Unnamed: 59,Unnamed: 60,Q2 2018,Unnamed: 62,Unnamed: 63,Unnamed: 64,Unnamed: 65,Unnamed: 66,Q3 2018,Unnamed: 68,Unnamed: 69,Unnamed: 70,Unnamed: 71,Unnamed: 72,Q4 2018,Unnamed: 74,Unnamed: 75,Unnamed: 76,Unnamed: 77,Unnamed: 78,Q1 2019,Unnamed: 80,Unnamed: 81,Unnamed: 82,Unnamed: 83,Unnamed: 84,Q2 2019,Unnamed: 86,Unnamed: 87,Unnamed: 88,Unnamed: 89,Unnamed: 90,Q3 2019,Unnamed: 92,Unnamed: 93,Unnamed: 94,Unnamed: 95,Unnamed: 96,Q4 2019,Unnamed: 98,Unnamed: 99,Unnamed: 100,Unnamed: 101,Unnamed: 102,Q1 2020,Unnamed: 104,Unnamed: 105,Unnamed: 106,Unnamed: 107,Unnamed: 108,Q2 2020,Unnamed: 110,Unnamed: 111,Unnamed: 112,Unnamed: 113,Unnamed: 114,Q3 2020,Unnamed: 116,Unnamed: 117,Unnamed: 118,Unnamed: 119,Unnamed: 120,Q4 2020,Unnamed: 122,Unnamed: 123,Unnamed: 124,Unnamed: 125,Unnamed: 126,Q1 2021,Unnamed: 128,Unnamed: 129,Unnamed: 130,Unnamed: 131,Unnamed: 132,Q2 2021,Unnamed: 134,Unnamed: 135,Unnamed: 136,Unnamed: 137,Unnamed: 138,Q3 2021,Unnamed: 140,Unnamed: 141,Unnamed: 142,Unnamed: 143,Unnamed: 144,Q4 2021,Unnamed: 146,Unnamed: 147,Unnamed: 148,Unnamed: 149,Unnamed: 150,Q1 2022,Unnamed: 152,Unnamed: 153,Unnamed: 154,Unnamed: 155,Unnamed: 156,Q2 2022,Unnamed: 158,Unnamed: 159,Unnamed: 160,Unnamed: 161,Unnamed: 162,Q3 2022,Unnamed: 164,Unnamed: 165,Unnamed: 166,Unnamed: 167,Unnamed: 168,Q4 2022,Unnamed: 170,Unnamed: 171,Unnamed: 172,Unnamed: 173,Unnamed: 174,Q1 2023,Unnamed: 176,Unnamed: 177,Unnamed: 178,Unnamed: 179,Unnamed: 180,Q2 2023,Unnamed: 182,Unnamed: 183,Unnamed: 184,Unnamed: 185,Unnamed: 186,Q3 2023,Unnamed: 188,Unnamed: 189,Unnamed: 190,Unnamed: 191,Unnamed: 192,Q4 2023,Unnamed: 194,Unnamed: 195,Unnamed: 196,Unnamed: 197,Unnamed: 198
0,NaN,Total Loan Book,NaN,Loans in Arrears over 90 days,NaN,Loans in Arrears over 720 days,NaN,Total Loan Book,NaN,Loans in Arrears over 90 days,NaN,Loans in Arrears over 720 days,NaN,Total Loan Book,NaN,Loans in Arrears over 90 days,NaN,Loans in Arrears over 720 days,NaN,Total Loan Book,NaN,Loans in Arrears over 90 days,NaN,Loans in Arrears over 720 days,NaN,Total Loan Book,NaN,Loans in Arrears over 90 days,NaN,Loans in Arrears over 720 days,NaN,Total Loan Book,NaN,Loans in Arrears over 90 days,NaN,Loans in Arrears over 720 days,NaN,Total Loan Book,NaN,Loans in Arrears over 90 days,NaN,Loans in Arrears over 720 days,NaN,Total Loan Book,NaN,Loans in Arrears over 90 days,NaN,Loans in Arrears over 720 days,NaN,Total Loan Book,NaN,Loans in Arrears over 90 days,NaN,Loans in Arrears over 720 days,NaN,Total Loan Book,NaN,Loans in Arrears over 90 days,NaN,Loans in Arrears over 720 days,NaN,Total Loan Book,NaN,Loans in Arrears over 90 days,NaN,Loans in Arrears over 720 days,NaN,Total Loan Book,NaN,Loans in Arrears over 90 days,NaN,Loans in Arrears over 720 days,NaN,Total Loan Book,NaN,Loans in Arrears over 90 days,NaN,Loans in Arrears over 720 days,NaN,Total Loan Book,NaN,Loans in Arrears over 90 days,NaN,Loans in Arrears over 720 days,NaN,Total Loan Book,NaN,Loans in Arrears over 90 days,NaN,Loans in Arrears over 720 days,NaN,Total Loan Book,NaN,Loans in Arrears over 90 days,NaN,Loans in Arrears over 720 days,NaN,Total Loan Book,NaN,Loans in Arrears over 90 days,NaN,Loans in Arrears over 720 days,NaN,Total Loan Book,NaN,Loans in Arrears over 90 days,NaN,Loans in Arrears over 720 days,NaN,Total Loan Book,NaN,Loans in Arrears over 90 days,NaN,L

In [26]:
# Row 0 contains metric groups like:
# Total Loan Book / Loans in Arrears over 90 days / Loans in Arrears over 720 days
# Row 1 contains subtypes like:
# Number / Value (€)

top_row = arrears_raw.iloc[0]
second_row = arrears_raw.iloc[1]

new_cols = []

current_quarter = None

for i, col in enumerate(arrears_raw.columns):
    col_name = str(col).strip()

    # first column should stay as local authority
    if i == 0:
        new_cols.append("local_authority")
        continue

    # detect quarter from original header row
    if re.match(r"Q[1-4]\s+\d{4}", col_name):
        current_quarter = col_name

    metric_group = str(top_row.iloc[i]).strip() if pd.notna(top_row.iloc[i]) else ""
    metric_sub = str(second_row.iloc[i]).strip() if pd.notna(second_row.iloc[i]) else ""

    # fill empty quarter from previous seen quarter
    if current_quarter is None:
        current_quarter = ""

    full_name = f"{current_quarter} | {metric_group} | {metric_sub}".strip(" |")
    new_cols.append(full_name)

new_cols[:20]

['local_authority',
 'Q4 2015 | Total Loan Book | Number',
 'Q4 2015 |  | Value (€)',
 'Q4 2015 | Loans in Arrears over 90 days | Number',
 'Q4 2015 |  | Value (€)',
 'Q4 2015 | Loans in Arrears over 720 days | Number',
 'Q4 2015 |  | Value (€)',
 'Q1 2016 | Total Loan Book | Number',
 'Q1 2016 |  | Value (€)',
 'Q1 2016 | Loans in Arrears over 90 days | Number',
 'Q1 2016 |  | Value (€)',
 'Q1 2016 | Loans in Arrears over 720 days | Number',
 'Q1 2016 |  | Value (€)',
 'Q2 2016 | Total Loan Book | Number',
 'Q2 2016 |  | Value (€)',
 'Q2 2016 | Loans in Arrears over 90 days | Number',
 'Q2 2016 |  | Value (€)',
 'Q2 2016 | Loans in Arrears over 720 days | Number',
 'Q2 2016 |  | Value (€)',
 'Q3 2016 | Total Loan Book | Number']

In [27]:
arrears = arrears_raw.copy()
arrears.columns = new_cols

# drop the two metadata rows used to build column names
arrears = arrears.iloc[2:].copy()

arrears.head()
print(arrears.columns.tolist()[:20])

['local_authority', 'Q4 2015 | Total Loan Book | Number', 'Q4 2015 |  | Value (€)', 'Q4 2015 | Loans in Arrears over 90 days | Number', 'Q4 2015 |  | Value (€)', 'Q4 2015 | Loans in Arrears over 720 days | Number', 'Q4 2015 |  | Value (€)', 'Q1 2016 | Total Loan Book | Number', 'Q1 2016 |  | Value (€)', 'Q1 2016 | Loans in Arrears over 90 days | Number', 'Q1 2016 |  | Value (€)', 'Q1 2016 | Loans in Arrears over 720 days | Number', 'Q1 2016 |  | Value (€)', 'Q2 2016 | Total Loan Book | Number', 'Q2 2016 |  | Value (€)', 'Q2 2016 | Loans in Arrears over 90 days | Number', 'Q2 2016 |  | Value (€)', 'Q2 2016 | Loans in Arrears over 720 days | Number', 'Q2 2016 |  | Value (€)', 'Q3 2016 | Total Loan Book | Number']


In [28]:
COUNTY_MAP = {
    "Carlow County Council": "Carlow",
    "Cavan County Council *": "Cavan",
    "Clare County Council": "Clare",
    "Cork City Council": "Cork",
    "Cork County Council": "Cork",
    "Donegal County Council": "Donegal",
    "Dublin City Council": "Dublin",
    "DLR County Council": "Dublin",
    "Fingal County Council 1": "Dublin",
    "Galway City Council": "Galway",
    "Galway County Council": "Galway",
    "Kerry County Council": "Kerry",
    "Kildare County Council": "Kildare",
    "Kilkenny County Council": "Kilkenny",
    "Laois County Council": "Laois",
    "Leitrim County Council": "Leitrim",
    "Limerick City and County Council": "Limerick",
    "Longford County Council": "Longford",
    "Louth County Council": "Louth",
    "Mayo County Council": "Mayo",
    "Meath County Council": "Meath",
    "Monaghan County Council": "Monaghan",
    "Offaly County Council": "Offaly",
    "Roscommon County Council": "Roscommon",
    "Sligo County Council": "Sligo",
    "Tipperary County Council": "Tipperary",
    "Waterford City and County Council": "Waterford",
    "Westmeath County Council": "Westmeath",
    "Wexford County Council": "Wexford",
    "Wicklow County Council": "Wicklow",
}

def standardize_county(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s in COUNTY_MAP:
        return COUNTY_MAP[s]
    s = s.replace(" County Council", "").replace(" City Council", "").strip()
    return s

arrears["area_name"] = arrears["local_authority"].map(standardize_county)

arrears[["local_authority", "area_name"]].head(10)

,local_authority,area_name
2,Carlow County Council,Carlow
3,Cavan County Council *,Cavan
4,Clare County Council,Clare
5,Cork City Council,Cork
6,Cork County Council,Cork
7,Donegal County Council,Donegal
8,Dublin City Council,Dublin
9,DLR County Council,Dublin
10,Fingal County Council 1,Dublin
11,Galway City Council,Galway


In [29]:
quarter_cols = [c for c in arrears.columns if re.match(r"Q[1-4]\s+\d{4}", str(c))]
print("Number of quartered metric columns:", len(quarter_cols))
print(quarter_cols[:15])

Number of quartered metric columns: 198
['Q4 2015 | Total Loan Book | Number', 'Q4 2015 |  | Value (€)', 'Q4 2015 | Loans in Arrears over 90 days | Number', 'Q4 2015 |  | Value (€)', 'Q4 2015 | Loans in Arrears over 720 days | Number', 'Q4 2015 |  | Value (€)', 'Q1 2016 | Total Loan Book | Number', 'Q1 2016 |  | Value (€)', 'Q1 2016 | Loans in Arrears over 90 days | Number', 'Q1 2016 |  | Value (€)', 'Q1 2016 | Loans in Arrears over 720 days | Number', 'Q1 2016 |  | Value (€)', 'Q2 2016 | Total Loan Book | Number', 'Q2 2016 |  | Value (€)', 'Q2 2016 | Loans in Arrears over 90 days | Number']


In [30]:
quarter_cols = [c for c in arrears.columns if re.match(r"Q[1-4]\s+\d{4}", str(c))]
print("Number of quartered metric columns:", len(quarter_cols))
print(quarter_cols[:15])

Number of quartered metric columns: 198
['Q4 2015 | Total Loan Book | Number', 'Q4 2015 |  | Value (€)', 'Q4 2015 | Loans in Arrears over 90 days | Number', 'Q4 2015 |  | Value (€)', 'Q4 2015 | Loans in Arrears over 720 days | Number', 'Q4 2015 |  | Value (€)', 'Q1 2016 | Total Loan Book | Number', 'Q1 2016 |  | Value (€)', 'Q1 2016 | Loans in Arrears over 90 days | Number', 'Q1 2016 |  | Value (€)', 'Q1 2016 | Loans in Arrears over 720 days | Number', 'Q1 2016 |  | Value (€)', 'Q2 2016 | Total Loan Book | Number', 'Q2 2016 |  | Value (€)', 'Q2 2016 | Loans in Arrears over 90 days | Number']


In [33]:
quarter_pattern = re.compile(r"(Q[1-4])\s+(\d{4})")

arrears_rows = []

for _, row in arrears.iterrows():
    area = row["area_name"]
    if pd.isna(area):
        continue

    # get all quarter prefixes present in column names
    quarter_prefixes = sorted({
        f"{m.group(1)} {m.group(2)}"
        for c in arrears.columns
        for m in [quarter_pattern.match(str(c))]
        if m
    })

    for prefix in quarter_prefixes:
        qm = re.match(r"(Q[1-4])\s+(\d{4})", prefix)
        quarter = int(qm.group(1)[1])
        year = int(qm.group(2))

        rec = {
            "area_name": area,
            "year": year,
            "quarter": quarter,
            "time_period": f"{year}Q{quarter}",
        }

        for c in arrears.columns:
            if not str(c).startswith(prefix):
                continue

            val = row[c]
            if isinstance(val, str):
                val = val.replace("€", "").replace(",", "").strip()
            val = pd.to_numeric(val, errors="coerce")

            cl = str(c).lower()

            if "total loan book" in cl and "number" in cl:
                rec["mortgage_loan_book_n"] = val
            elif "total loan book" in cl and "value" in cl:
                rec["mortgage_loan_book_value"] = val
            elif "loans in arrears over 90 days" in cl and "number" in cl:
                rec["arrears_90d_n"] = val
            elif "loans in arrears over 90 days" in cl and "value" in cl:
                rec["arrears_90d_value"] = val
            elif "loans in arrears over 720 days" in cl and "number" in cl:
                rec["arrears_720d_n"] = val
            elif "loans in arrears over 720 days" in cl and "value" in cl:
                rec["arrears_720d_value"] = val

        arrears_rows.append(rec)

arrears_long = pd.DataFrame(arrears_rows)
arrears_long.head()

,area_name,year,quarter,time_period,mortgage_loan_book_n,arrears_90d_n,arrears_720d_n
0,Carlow,2016,1,2016Q1,241.0,35.0,4.0
1,Carlow,2017,1,2017Q1,240.0,32.0,5.0
2,Carlow,2018,1,2018Q1,250.0,35.0,7.0
3,Carlow,2019,1,2019Q1,248.0,21.0,4.0
4,Carlow,2020,1,2020Q1,258.0,29.0,8.0


In [34]:
print("Signals periods:", signals["time_period"].unique()[:10])
print("Arrears periods:", arrears_long["time_period"].unique()[:10])

Signals periods: <StringArray>
['2025Q3', '2025Q2', '2025Q1', '2024Q4', '2024Q3', '2024Q2', '2024Q1',
 '2023Q4', '2023Q3', '2023Q2']
Length: 10, dtype: str
Arrears periods: <StringArray>
['2016Q1', '2017Q1', '2018Q1', '2019Q1', '2020Q1', '2021Q1', '2022Q1',
 '2023Q1', '2016Q2', '2017Q2']
Length: 10, dtype: str


In [35]:
df = (
    signals
    .merge(
        rents_q,
        on=["area_name", "year", "quarter", "time_period"],
        how="left"
    )
    .merge(
        ppi_q,
        on=["year", "quarter", "time_period"],   # national / Dublin context only
        how="left"
    )
    .merge(
        arrears_long[[
            "area_name", "year", "quarter", "time_period",
            "mortgage_loan_book_n",
            "arrears_90d_n",
            "arrears_720d_n"
        ]],
        on=["area_name", "year", "quarter", "time_period"],
        how="left"
    )
)

df.head()

,area_name,year,quarter,time_period,rent_growth,population_growth,housing_completions,rent_level,ppi_dublin_all,ppi_nat_ex_dublin_all,ppi_national_all,mortgage_loan_book_n,arrears_90d_n,arrears_720d_n
0,Mayo,2025,3,2025Q3,12.538864,2.016129,555.934208,NaN,38.316667,43.416667,42.625,NaN,NaN,NaN
1,Tipperary,2025,3,2025Q3,11.971680,1.554692,369.658378,NaN,38.316667,43.416667,42.625,NaN,NaN,NaN
2,Limerick,2025,3,2025Q3,13.607647,1.381852,560.241862,NaN,38.316667,43.416667,42.625,NaN,NaN,NaN
3,Carlow,2025,3,2025Q3,12.420807,2.180685,717.817791,NaN,38.316667,43.416667,42.625,NaN,NaN,NaN
4,Sligo,2025,3,2025Q3,12.957081,0.413793,431.541413,NaN,38.316667,43.416667,42.625,NaN,NaN,NaN


In [36]:
df["arrears_90d_rate"] = (
    df["arrears_90d_n"] / df["mortgage_loan_book_n"].replace(0, np.nan)
)

df["arrears_720d_rate"] = (
    df["arrears_720d_n"] / df["mortgage_loan_book_n"].replace(0, np.nan)
)

In [37]:
print(df.shape)

print(
    df[[
        "rent_level",
        "ppi_national_all",
        "mortgage_loan_book_n",
        "arrears_90d_n"
    ]].isna().mean().sort_values()
)

(902, 16)
ppi_national_all        0.000000
rent_level              0.086475
mortgage_loan_book_n    0.201774
arrears_90d_n           0.201774
dtype: float64


In [42]:
print(df.shape)

print(
    df[[
        "rent_level",
        "ppi_national_all",
        "mortgage_loan_book_n",
        "arrears_90d_n"
    ]].isna().mean().sort_values()
)

(902, 16)
rent_level              0.0
ppi_national_all        0.0
mortgage_loan_book_n    0.0
arrears_90d_n           0.0
dtype: float64


In [39]:
df = df.sort_values(["area_name", "year", "quarter"]).copy()

df[[
    "mortgage_loan_book_n",
    "arrears_90d_n",
    "arrears_720d_n"
]] = df.groupby("area_name")[[
    "mortgage_loan_book_n",
    "arrears_90d_n",
    "arrears_720d_n"
]].ffill()

In [41]:
df["rent_level"] = df.groupby("area_name")["rent_level"].ffill()

In [43]:
df.head()

,area_name,year,quarter,time_period,rent_growth,population_growth,housing_completions,rent_level,ppi_dublin_all,ppi_nat_ex_dublin_all,ppi_national_all,mortgage_loan_book_n,arrears_90d_n,arrears_720d_n,arrears_90d_rate,arrears_720d_rate
900,Carlow,2018,1,2018Q1,7.174631,1.215278,855.082714,774.45,29.358333,27.108333,29.141667,250.0,35.0,7.0,0.140000,0.028000
870,Carlow,2018,2,2018Q2,5.710050,1.215278,855.082714,774.45,29.283333,29.025000,30.066667,250.0,32.0,7.0,0.128000,0.028000
835,Carlow,2018,3,2018Q3,7.238179,1.215278,855.082714,774.45,29.058333,28.408333,29.708333,252.0,28.0,7.0,0.111111,0.027778
803,Carlow,2018,4,2018Q4,6.610302,1.215278,855.082714,774.45,27.983333,27.691667,28.808333,250.0,25.0,6.0,0.100000,0.024000
761,Carlow,2019,1,2019Q1,6.498452,1.886792,855.082714,826.31,26.150000,26.800000,27.441667,248.0,21.0,4.0,0.084677,0.016129
